# Step 4: Category Mapping + Localization
Map the 434 PCard merchant categories to Masroof's 6 categories (drop what fits none), keep the top 20 agencies as departments, create integer user IDs with Arabic names in English letters (+ 20 managers, 2 finance admins), convert amounts to SAR (×3.75), shift dates 626 weeks − 1 day (Fri–Sat weekend), generate descriptions.

**Input:** `data/processed/pcard_clean.csv`  
**Output:** `data/processed/requests.csv`, `departments.csv`, `users.csv`, and `mappings/mcc_to_category.csv`

In [4]:
import pandas as pd
import numpy as np
import os
import re

df = pd.read_csv("../data/processed/pcard_clean.csv",
                 parse_dates=["transaction_date", "posted_date"])

text_cols = ["description", "vendor", "mcc"]
df[text_cols] = df[text_cols].fillna("UNKNOWN")
print("Loaded:", df.shape)

Loaded: (427872, 12)


In [5]:
CATEGORY_RULES = {
    "Travel": (
        r"AIRLINE|AIR LINES|AIRWAYS|AIR CARRIER|TRAVEL AGENC|\bDELTA\b|"
        r"UNITED AIRLINES|SOUTHWEST|AMERICAN AIRLINES|HOTEL|MOTEL|\bINNS?\b|"
        r"LODGING|RESORT|SUITES|MARRIOTT|HILTON|HYATT|SHERATON|WESTIN|RADISSON|"
        r"EMBASSY|DOUBLETREE|HAMPTON|COURTYARD|RAMADA|BEST WESTERN|LA QUINTA|"
        r"COMFORT|WYNDHAM|CROWNE|RENAISSANCE|\bOMNI\b"
    ),
    "Transportation": (
        r"TAXI|LIMOUSINE|PARKING|AUTOMATED FUEL|SERVICE STATION|CAR RENTAL|"
        r"RENT-A-CAR|RENTAL CAR|AUTO RENT|BUS LINES|TOLLS|RAILROAD|PASSENGER|"
        r"COMMUTER|AUTOMOTIVE|\bTIRE|CAR WASH|TOWING"
    ),
    "Food": (
        r"GROCERY|RESTAURANT|\bEATING\b|FAST FOOD|CATERER|BAKER|"
        r"MISCELLANEOUS FOOD|CANDY"
    ),
    "Office Supplies": (
        r"STATIONERY|OFFICE SUPPL|OFFICE AND SCHOOL|OFFICE, PHOTOGRAPHIC|PRINTING|"
        r"BOOK STORES|BOOKS, PERIODICALS|NEWS DEALERS|QUICK-COPY"
    ),
    "IT & Telecom": (
        r"COMPUTER|SOFTWARE|TELECOMMUNICATION|TELEPHONE|CABLE, SATELLITE|"
        r"ELECTRONICS STORES|DATA PROCESS|INFORMATION RETRIEVAL"
    ),
    "Equipment & Maintenance": (
        r"INDUSTRIAL|HARDWARE|HOME SUPPLY|COMMERCIAL EQUIPMENT|LUMBER|"
        r"BUILDING MATERIALS|PLUMBING|ELECTRICAL|PAINT|\bTOOL|MACHINERY|"
        r"CONTRACTOR|HEATING|GLASS|NURSER|LAWN|JANITOR|CLEANING|CHEMICAL|"
        r"METAL SERVICE|CONSTRUCTION MATERIALS"
    ),
}

def to_category(mcc):
    for category, pattern in CATEGORY_RULES.items():
        if re.search(pattern, mcc):
            return category
    return None   # fits none of the 6

In [6]:
mcc_table = (df["mcc"].value_counts()
               .rename_axis("mcc")
               .reset_index(name="rows"))
mcc_table["category"] = mcc_table["mcc"].apply(to_category)
print("MCCs:", len(mcc_table), "| mapped:", mcc_table["category"].notna().sum())

os.makedirs("../mappings", exist_ok=True)
mcc_table.to_csv("../mappings/mcc_to_category.csv", index=False)

MCCs: 434 | mapped: 199


In [7]:
summary = (mcc_table.groupby("category", dropna=False)["rows"].sum()
                     .sort_values(ascending=False))
print((summary / len(df) * 100).round(1))

for cat in CATEGORY_RULES:
    print(f"\n{cat}")
    print(mcc_table[mcc_table["category"] == cat].head(5).to_string(index=False))

category
NaN                        34.0
Equipment & Maintenance    19.2
Office Supplies            15.6
Travel                     13.6
IT & Telecom                7.5
Food                        6.2
Transportation              4.1
Name: rows, dtype: float64

Travel
                      mcc  rows category
          TRAVEL AGENCIES 11542   Travel
        AMERICAN AIRLINES 11086   Travel
AIRLINES AND AIR CARRIERS  6125   Travel
       SOUTHWEST AIRLINES  3803   Travel
          UNITED AIRLINES  3169   Travel

Transportation
                                                 mcc  rows       category
             AUTOMOTIVE PARTS AND ACCESSORIES STORES  6848 Transportation
MISC. AUTOMOTIVE,AIRCRAFT,AND FARM EQUIPMENT DEALERS  2114 Transportation
                            AUTOMATED FUEL DISPENSER  1906 Transportation
SERVICE STATIONS (WITH OR WITHOUT ANCILLARY SERVICES  1672 Transportation
                              AUTOMOTIVE TIRE STORES  1106 Transportation

Food
                    

In [8]:
mapping = dict(zip(mcc_table["mcc"], mcc_table["category"]))
df["category"] = df["mcc"].map(mapping)

before = len(df)
df = df[df["category"].notna()].copy()
print(f"Kept {len(df):,} of {before:,} rows ({len(df)/before:.1%})")
print("Employees:", df["cardholder_id"].nunique(), "| Agencies:", df["agency_number"].nunique())
print(df["category"].value_counts())

Kept 282,502 of 427,872 rows (66.0%)
Employees: 5217 | Agencies: 113
category
Equipment & Maintenance    81992
Office Supplies            66757
Travel                     58022
IT & Telecom               31983
Food                       26380
Transportation             17368
Name: count, dtype: int64


In [9]:
N_DEPARTMENTS = 20

dept_rows = df["agency_number"].value_counts()
top_agencies = dept_rows.head(N_DEPARTMENTS).index.tolist()

before = len(df)
df = df[df["agency_number"].isin(top_agencies)].copy()
print(f"Kept {len(df):,} of {before:,} rows ({len(df)/before:.1%})")

Kept 247,838 of 282,502 rows (87.7%)


In [10]:
DEPARTMENT_NAMES = {
    1000:  "Operations",
    76000: "Business Development",
    77000: "Research & Development",
    13100: "Facilities Management",
    34500: "Fleet & Logistics",
    56600: "Projects",
    98000: "Engineering",
    80500: "Field Services",
    66500: "Hospitality & Events",
    58500: "Security",
    9000:  "Administration",
    2500:  "Maintenance",
    65000: "Health & Safety",
    45200: "Human Resources",
    32000: "Environment & Sustainability",
    53000: "Training & Development",
    75000: "Sales",
    4000:  "Information Technology",
    80000: "Marketing",
    42000: "Public Relations",
}

departments = pd.DataFrame({"agency_number": top_agencies})
departments["department_id"] = range(1, len(departments) + 1)
departments["name"] = departments["agency_number"].map(DEPARTMENT_NAMES)
assert departments["name"].notna().all(), "An agency in the top N has no department name"

df["department_id"] = df["agency_number"].map(
    dict(zip(departments["agency_number"], departments["department_id"])))
print(departments.to_string(index=False))

 agency_number  department_id                         name
          1000              1                   Operations
         76000              2         Business Development
         77000              3       Research & Development
         13100              4        Facilities Management
         34500              5            Fleet & Logistics
         56600              6                     Projects
         98000              7                  Engineering
         80500              8               Field Services
         66500              9         Hospitality & Events
         58500             10                     Security
          9000             11               Administration
          2500             12                  Maintenance
         65000             13              Health & Safety
         45200             14              Human Resources
         32000             15 Environment & Sustainability
         53000             16       Training & Developme

In [12]:
df.head()

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,mcc,cardholder_id,category,department_id
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,2.80,PIKEPASS REBILLS,2013-07-01,2013-07-02,TOLLS AND BRIDGE FEES,1000_ABBOTT_T,Transportation,1
1,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,AIR TRAVEL,297.60,DELTA AIR 0062335020780,2013-07-05,2013-07-08,DELTA,1000_ABBOTT_T,Travel,1
2,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,068934075 PCS,362.37,XEROX CORPORATION/RBO,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,Office Supplies,1
3,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,1068.55,BUSINESS IMAGING SYSTEMS,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,Office Supplies,1
9,201308,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,2.80,PIKEPASS REBILLS,2013-08-01,2013-08-02,TOLLS AND BRIDGE FEES,1000_ABBOTT_T,Transportation,1


In [13]:
employees = (df[["cardholder_id", "department_id"]]
               .drop_duplicates()
               .sort_values("cardholder_id")
               .reset_index(drop=True))
employees["user_id"] = range(1001, 1001 + len(employees))

df["requester_id"] = df["cardholder_id"].map(
    dict(zip(employees["cardholder_id"], employees["user_id"])))
print("Employees:", len(employees), "| user ids", employees["user_id"].min(), "to", employees["user_id"].max())

Employees: 4577 | user ids 1001 to 5577


In [14]:
FIRST_NAMES = [
    "Abdullah", "Mohammed", "Ahmed", "Fahad", "Khalid", "Saud", "Faisal", "Sultan",
    "Nasser", "Turki", "Bandar", "Majed", "Nawaf", "Yousef", "Omar", "Ali",
    "Hassan", "Ibrahim", "Saleh", "Abdulrahman", "Abdulaziz", "Hamad", "Rayan", "Ziyad",
    "Mazen", "Waleed", "Talal", "Badr", "Meshal", "Osama", "Yazeed", "Anas",
    "Nora", "Sara", "Reem", "Lama", "Hessa", "Maha", "Amal", "Haifa",
    "Dalal", "Nouf", "Ghada", "Asma", "Hind", "Mashael", "Rawan", "Shahad",
    "Alanoud", "Jawaher", "Lujain", "Razan", "Wafa", "Bashayer", "Afnan", "Dana",
]
FAMILY_NAMES = [
    "Al-Harbi", "Al-Otaibi", "Al-Qahtani", "Al-Ghamdi", "Al-Zahrani", "Al-Shehri",
    "Al-Dosari", "Al-Mutairi", "Al-Anazi", "Al-Shammari", "Al-Malki", "Al-Subaie",
    "Al-Juhani", "Al-Omari", "Al-Amri", "Al-Asmari", "Al-Hazmi", "Al-Faifi",
    "Al-Rashidi", "Al-Khaldi", "Al-Yami", "Al-Sulami", "Al-Bishi", "Al-Tamimi",
    "Al-Hakami", "Al-Madkhali", "Al-Abdali", "Al-Saadi", "Al-Jaber", "Al-Hamdan",
    "Al-Ahmadi", "Al-Mansour", "Al-Najjar", "Al-Khathami", "Al-Sahli", "Al-Muqrin",
    "Al-Shahrani", "Al-Qarni", "Al-Balawi", "Al-Enezi", "Al-Ruwaili", "Al-Thubaiti",
]

rng = np.random.default_rng(42)

def make_people(n):
    first = rng.choice(FIRST_NAMES, size=n)
    last = rng.choice(FAMILY_NAMES, size=n)
    return [f"{f} {l}" for f, l in zip(first, last)]

employees["full_name"] = make_people(len(employees))
employees["role"] = "employee"

In [15]:
managers = departments[["department_id"]].copy()
managers["user_id"] = range(101, 101 + len(managers))
managers["full_name"] = make_people(len(managers))
managers["role"] = "manager"
departments["manager_id"] = managers["user_id"].values

finance = pd.DataFrame({
    "user_id": [201, 202],
    "full_name": make_people(2),
    "role": "finance_admin",
    "department_id": pd.NA,
})

In [16]:
users = pd.concat([
    managers[["user_id", "full_name", "role", "department_id"]],
    finance[["user_id", "full_name", "role", "department_id"]],
    employees[["user_id", "full_name", "role", "department_id"]],
], ignore_index=True)
users["department_id"] = users["department_id"].astype("Int64")

base = (users["full_name"].str.lower()
                          .str.replace("al-", "al", regex=False)
                          .str.replace(" ", ".", regex=False))
users["username"] = base + "." + users["user_id"].astype(str)
users["email"] = users["username"] + "@example.com"

assert users["user_id"].is_unique and users["username"].is_unique
print(users["role"].value_counts())
print(users.sample(5, random_state=1).to_string(index=False))
print("Distinct full names:", users["full_name"].nunique(), "of", len(users))

role
employee         4577
manager            20
finance_admin       2
Name: count, dtype: int64
 user_id             full_name     role  department_id                  username                                 email
    2330 Abdulrahman Al-Khaldi employee              1 abdulrahman.alkhaldi.2330 abdulrahman.alkhaldi.2330@example.com
    2666       Rayan Al-Balawi employee              1       rayan.albalawi.2666       rayan.albalawi.2666@example.com
    2276        Haifa Al-Malki employee              1        haifa.almalki.2276        haifa.almalki.2276@example.com
    3080       Sultan Al-Qarni employee              4       sultan.alqarni.3080       sultan.alqarni.3080@example.com
    3297        Turki Al-Omari employee             15        turki.alomari.3297        turki.alomari.3297@example.com
Distinct full names: 2032 of 4599


In [17]:
df.head()

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,mcc,cardholder_id,category,department_id,requester_id
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,2.80,PIKEPASS REBILLS,2013-07-01,2013-07-02,TOLLS AND BRIDGE FEES,1000_ABBOTT_T,Transportation,1,1001
1,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,AIR TRAVEL,297.60,DELTA AIR 0062335020780,2013-07-05,2013-07-08,DELTA,1000_ABBOTT_T,Travel,1,1001
2,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,068934075 PCS,362.37,XEROX CORPORATION/RBO,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,Office Supplies,1,1001
3,201307,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,1068.55,BUSINESS IMAGING SYSTEMS,2013-07-12,2013-07-15,"OFFICE, PHOTOGRAPHIC, PHOTOCOPY, AND MICROFILM...",1000_ABBOTT_T,Office Supplies,1,1001
9,201308,1000,OKLAHOMA STATE UNIVERSITY,ABBOTT,T,GENERAL PURCHASE,2.80,PIKEPASS REBILLS,2013-08-01,2013-08-02,TOLLS AND BRIDGE FEES,1000_ABBOTT_T,Transportation,1,1001


In [18]:
USD_TO_SAR = 3.75
df["amount_usd"] = df["amount"]
df["amount_sar"] = (df["amount"] * USD_TO_SAR).round(2)
print(df[["amount_usd", "amount_sar"]].describe().round(2))

       amount_usd  amount_sar
count   247838.00   247838.00
mean       407.43     1527.86
std       4324.43    16216.61
min          0.01        0.04
25%         30.98      116.18
50%         96.88      363.30
75%        313.11     1174.15
max    1089180.00  4084425.00


In [19]:
DATE_SHIFT = pd.Timedelta(weeks=626) - pd.Timedelta(days=1)
df["expense_date"] = df["transaction_date"] + DATE_SHIFT

print("Old range:", df["transaction_date"].min().date(), "to", df["transaction_date"].max().date())
print("New range:", df["expense_date"].min().date(), "to", df["expense_date"].max().date())

check = pd.DataFrame({
    "old_day": df["transaction_date"].dt.day_name(),
    "new_day": df["expense_date"].dt.day_name(),
}).value_counts().reset_index(name="rows")
print(check.to_string(index=False))

Old range: 2013-05-16 to 2014-06-30
New range: 2025-05-14 to 2026-06-28
  old_day   new_day  rows
Wednesday   Tuesday 50109
 Thursday Wednesday 48935
  Tuesday    Monday 48118
   Friday  Thursday 45147
   Monday    Sunday 35213
 Saturday    Friday 13682
   Sunday  Saturday  6634


In [20]:
df = df.sort_values(["expense_date", "requester_id", "amount_sar"], kind="mergesort").reset_index(drop=True)

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,mcc,cardholder_id,category,department_id,requester_id,amount_usd,amount_sar,expense_date
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ROPERS,A,ROOM CHARGES,15120.00,BETHESDA COURT HOTEL,2013-05-16,2013-07-24,"LODGING--HOTELS,MOTELS,AND RESORTS",1000_ROPERS_A,Travel,1,2450,15120.00,56700.00,2025-05-14
1,201307,13100,DEPARTMENT OF CORRECTIONS,GULLEY,J,GENERAL PURCHASE,88.94,QT 40 01000405,2013-06-18,2013-07-02,AUTOMATED FUEL DISPENSER,13100_GULLEY_J,Transportation,4,2978,88.94,333.52,2025-06-16
2,201307,76000,UNIVERSITY OF OKLAHOMA,GRALL,N,GENERAL PURCHASE,2061.86,INVENSYS SYSTEMS INC,2013-06-19,2013-07-12,"COMPUTERS, COMPUTER PERIPHERAL EQUIPMENT, SOFT...",76000_GRALL_N,IT & Telecom,2,4399,2061.86,7731.98,2025-06-17
3,201307,80000,OKLA. CAREER AND TECHNOLOGY EDUCATION,FOOTE,R,ROOM CHARGES,6.16,LA QUINTA STILLWATER,2013-06-19,2013-07-01,LA QUINTA MOTOR INNS,80000_FOOTE_R,Travel,19,5351,6.16,23.10,2025-06-17
4,201307,76000,UNIVERSITY OF OKLAHOMA,WHEELER,T,GENERAL PURCHASE,6.12,MORRISON SUPPLY #82,2013-06-21,2013-07-01,PLUMBING AND HEATING EQUIPMENT AND SUPPLIES,76000_WHEELER_T,Equipment & Maintenance,2,4888,6.12,22.95,2025-06-19


In [23]:
DESCRIPTIONS = {
    "Travel": ["Flight booking for business trip", "Hotel stay for business trip",
               "Travel for client meeting", "Conference travel", "Accommodation for site visit"],
    "Transportation": ["Fuel for company vehicle", "Vehicle maintenance", "Car rental for site visit",
                       "Parking and tolls", "Vehicle spare parts"],
    "Food": ["Team lunch", "Refreshments for meeting", "Office pantry supplies",
             "Catering for event", "Client dinner"],
    "Office Supplies": ["Office stationery", "Printer paper and toner", "Printing services",
                        "Books and reference materials", "Office supplies restock"],
    "IT & Telecom": ["Mobile and internet bill", "Software license", "Computer accessories",
                     "IT equipment purchase", "Network services"],
    "Equipment & Maintenance": ["Maintenance supplies", "Tools and hardware", "Cleaning supplies",
                                "Electrical parts", "Building repair materials"],
}
assert set(DESCRIPTIONS) == set(CATEGORY_RULES)

desc_rng = np.random.default_rng(7)
df["description"] = [DESCRIPTIONS[c][i] for c, i in
                     zip(df["category"], desc_rng.integers(0, 5, size=len(df)))]

In [24]:
df.head()

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,mcc,cardholder_id,category,department_id,requester_id,amount_usd,amount_sar,expense_date
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ROPERS,A,Accommodation for site visit,15120.00,BETHESDA COURT HOTEL,2013-05-16,2013-07-24,"LODGING--HOTELS,MOTELS,AND RESORTS",1000_ROPERS_A,Travel,1,2450,15120.00,56700.00,2025-05-14
1,201307,13100,DEPARTMENT OF CORRECTIONS,GULLEY,J,Parking and tolls,88.94,QT 40 01000405,2013-06-18,2013-07-02,AUTOMATED FUEL DISPENSER,13100_GULLEY_J,Transportation,4,2978,88.94,333.52,2025-06-16
2,201307,76000,UNIVERSITY OF OKLAHOMA,GRALL,N,IT equipment purchase,2061.86,INVENSYS SYSTEMS INC,2013-06-19,2013-07-12,"COMPUTERS, COMPUTER PERIPHERAL EQUIPMENT, SOFT...",76000_GRALL_N,IT & Telecom,2,4399,2061.86,7731.98,2025-06-17
3,201307,80000,OKLA. CAREER AND TECHNOLOGY EDUCATION,FOOTE,R,Accommodation for site visit,6.16,LA QUINTA STILLWATER,2013-06-19,2013-07-01,LA QUINTA MOTOR INNS,80000_FOOTE_R,Travel,19,5351,6.16,23.10,2025-06-17
4,201307,76000,UNIVERSITY OF OKLAHOMA,WHEELER,T,Cleaning supplies,6.12,MORRISON SUPPLY #82,2013-06-21,2013-07-01,PLUMBING AND HEATING EQUIPMENT AND SUPPLIES,76000_WHEELER_T,Equipment & Maintenance,2,4888,6.12,22.95,2025-06-19


In [25]:
time_rng = np.random.default_rng(11)
minutes = time_rng.integers(8 * 60, 17 * 60, size=len(df))
df["created_at"] = df["expense_date"] + pd.to_timedelta(minutes, unit="min")

df["expense_id"] = range(100001, 100001 + len(df))
df["status"] = "Approved"

In [26]:
df.head()

,year_month,agency_number,agency_name,last_name,first_initial,description,amount,vendor,transaction_date,posted_date,...,cardholder_id,category,department_id,requester_id,amount_usd,amount_sar,expense_date,created_at,expense_id,status
0,201307,1000,OKLAHOMA STATE UNIVERSITY,ROPERS,A,Accommodation for site visit,15120.00,BETHESDA COURT HOTEL,2013-05-16,2013-07-24,...,1000_ROPERS_A,Travel,1,2450,15120.00,56700.00,2025-05-14,2025-05-14 09:12:00,100001,Approved
1,201307,13100,DEPARTMENT OF CORRECTIONS,GULLEY,J,Parking and tolls,88.94,QT 40 01000405,2013-06-18,2013-07-02,...,13100_GULLEY_J,Transportation,4,2978,88.94,333.52,2025-06-16,2025-06-16 09:09:00,100002,Approved
2,201307,76000,UNIVERSITY OF OKLAHOMA,GRALL,N,IT equipment purchase,2061.86,INVENSYS SYSTEMS INC,2013-06-19,2013-07-12,...,76000_GRALL_N,IT & Telecom,2,4399,2061.86,7731.98,2025-06-17,2025-06-17 15:10:00,100003,Approved
3,201307,80000,OKLA. CAREER AND TECHNOLOGY EDUCATION,FOOTE,R,Accommodation for site visit,6.16,LA QUINTA STILLWATER,2013-06-19,2013-07-01,...,80000_FOOTE_R,Travel,19,5351,6.16,23.10,2025-06-17,2025-06-17 12:29:00,100004,Approved
4,201307,76000,UNIVERSITY OF OKLAHOMA,WHEELER,T,Cleaning supplies,6.12,MORRISON SUPPLY #82,2013-06-21,2013-07-01,...,76000_WHEELER_T,Equipment & Maintenance,2,4888,6.12,22.95,2025-06-19,2025-06-19 13:18:00,100005,Approved


In [27]:
requests = df.rename(columns={"vendor": "orig_vendor", "mcc": "orig_mcc"})[[
    "expense_id", "requester_id", "department_id", "category",
    "amount_sar", "amount_usd", "expense_date", "description",
    "status", "created_at", "orig_vendor", "orig_mcc",
]]
print(requests.head(3).to_string(index=False))

requests.to_csv("../data/processed/requests.csv", index=False)
departments[["department_id", "name", "manager_id", "agency_number"]].to_csv(
    "../data/processed/departments.csv", index=False)
users.to_csv("../data/processed/users.csv", index=False)
print("Saved requests:", requests.shape, "| departments:", len(departments), "| users:", len(users))

 expense_id  requester_id  department_id       category  amount_sar  amount_usd expense_date                  description   status          created_at          orig_vendor                                           orig_mcc
     100001          2450              1         Travel    56700.00    15120.00   2025-05-14 Accommodation for site visit Approved 2025-05-14 09:12:00 BETHESDA COURT HOTEL                 LODGING--HOTELS,MOTELS,AND RESORTS
     100002          2978              4 Transportation      333.52       88.94   2025-06-16            Parking and tolls Approved 2025-06-16 09:09:00       QT 40 01000405                           AUTOMATED FUEL DISPENSER
     100003          4399              2   IT & Telecom     7731.98     2061.86   2025-06-17        IT equipment purchase Approved 2025-06-17 15:10:00 INVENSYS SYSTEMS INC COMPUTERS, COMPUTER PERIPHERAL EQUIPMENT, SOFTWARE
Saved requests: (247838, 12) | departments: 20 | users: 4599


In [28]:
print("Rows:", len(requests), "| employees:", requests["requester_id"].nunique(),
      "| departments:", requests["department_id"].nunique())

weekend = requests["expense_date"].dt.dayofweek.isin([4, 5]).mean()
print(f"Weekend (Fri-Sat) share: {weekend:.1%}")

print(requests.groupby("category")["amount_sar"]
              .quantile([0.5, 0.95, 0.99]).unstack().round(0)
              .rename(columns={0.5: "median", 0.95: "p95", 0.99: "p99"}))

Rows: 247838 | employees: 4577 | departments: 20
Weekend (Fri-Sat) share: 8.2%
                         median     p95      p99
category                                        
Equipment & Maintenance   511.0  7588.0  18737.0
Food                      173.0  1744.0   4948.0
IT & Telecom              412.0  7140.0  20890.0
Office Supplies           267.0  2931.0   9345.0
Transportation            251.0  5307.0  17626.0
Travel                    708.0  3703.0   8573.0


C:\Users\Zala\AppData\Local\Temp\ipykernel_26612\633965712.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  weekend = requests["expense_date"].dt.dayofweek.isin([4, 5]).mean()


In [29]:
SUGGESTED_LIMITS = {
    "Office Supplies": 3000,
    "IT & Telecom": 7500,
    "Equipment & Maintenance": 7500,
    "Food": 2000,
    "Transportation": 5000,
    "Travel": 10000,
}
over = requests["amount_sar"] > requests["category"].map(SUGGESTED_LIMITS)
print(requests.assign(over=over).groupby("category")["over"].mean().mul(100).round(1))
print(f"Overall: {over.mean():.1%} of requests are over their category's limit")

category
Equipment & Maintenance    5.1
Food                       4.1
IT & Telecom               4.7
Office Supplies            4.8
Transportation             5.3
Travel                     0.8
Name: over, dtype: float64
Overall: 4.1% of requests are over their category's limit
